In [1]:
import util
import os
import json
import re
from pathlib import Path
import util.util as util

util.init_server()

Get all PDF files and generate chunks...
Embedding and FAISS Indexing...
SQLite and FTS5 tables setup...
Tables and FTS5 index created successfully!
Inserted 1175 rows into documents table
FTS5 index table populated


In [2]:
query_texts = [
    "QueryGym",
    "What is SRPO?",
    "Uncertainty-Guided Lookback",
    "Mixture-of-Experts Multimodal Large Language Models",
    "Uncooperative Behaviors in LLM based Multi-Agent Systems",
    "What is CARE-RAG?",
    "Liars' Bench",
    "What is TS-PEFT?",
    "SemanticCite?",
    "Training-Free One-Shot Federated Adaptation?"
]
for id, query_text in enumerate(query_texts, 1):
    print(f"\nQuery {id}: {query_text}\n")
    distances, indexes = util.query_faiss(query_text)
    results1 = []
    for i, idx in enumerate(indexes):
        results1.append(str(idx) + '|' + str(distances[i]))
    print(f"Vector Search Answer:\n{'\n'.join(results1)}\n")

    rows = util.query_keyword(query_text)
    results2 = []
    for row in rows:
        results2.append(str(row['id']) + '|' + str(row['rank']))
    print(f"Keyword Search Answer:\n{'\n'.join(results2)}\n")

    score_results3, results3 = util.query_hybrid(query_text, 3)
    
    print(f"Hybrid Search Answer:\n{'\n'.join(score_results3)}\n")

    hit1 = 0
    hit2 = 0
    for score_result in score_results3:
        if re.search(r'^vector\|[0-9]+\|keyword\|[0-9]+', score_result):
            hit1 +=1
            hit2 += 1
        elif re.search(r'^vector\|[0-9]+', score_result):
            hit1 +=1
        elif re.search(r'^keyword\|[0-9]+', score_result):
            hit2 +=1
    print(f"Vector hit: {hit1}/3    Keyword hit: {hit2}/3\n")


Query 1: QueryGym

Vector Search Answer:
315|1.0228137
317|1.0592775
316|1.1293777

Keyword Search Answer:
318|-10.318692716396537
319|-10.19553859140783
317|-10.094970541856522

Hybrid Search Answer:
vector|317|keyword|318|-3.4919107
vector|316|keyword|317|-3.3603616
keyword|319|-3.3251612

Vector hit: 2/3    Keyword hit: 3/3


Query 2: What is SRPO?

Vector Search Answer:
713|1.2985842
19|1.3371884
515|1.3508327

Keyword Search Answer:
21|-5.601506745794035

k_score for (713|1.298584222793579) not found in keyword_scores map, using -5.601506745794035
k_score for (19|1.3371883630752563) not found in keyword_scores map, using -5.601506745794035
k_score for (515|1.3508327007293701) not found in keyword_scores map, using -5.601506745794035
v_score for (21|-5.601506745794035) not found in faiss_scores map, using 1.8179759979248047
Hybrid Search Answer:
vector|713|-1.4614522
vector|19|-1.4382896
vector|515|-1.4301031

Vector hit: 3/3    Keyword hit: 0/3


Query 3: Uncertainty-Guided Lookb